In [65]:
import pandas as pd
from Bio import Entrez
Entrez.email = 'dmalzl@cemm.oeaw.ac.at'
Entrez.api_key = '746c80043a6e2ceeb01ae587f8ab1a8ace09'


srauid_map = pd.read_csv(
    'archs4_accession_to_uid.tsv',
    sep = '\t'
)

# may contain duplicates
srauid_map.drop_duplicates(inplace = True)
srauid_map

,accession,uid,database
0,SRX185895,243394,sra
1,SRX185896,243395,sra
2,SRX185897,243396,sra
3,SRX185898,243397,sra
4,SRX185899,243398,sra
...,...,...,...
163235,SRX3683704,5089665,sra
163236,SRX3683705,5089666,sra
163237,SRX3683706,5089667,sra
163238,SRX3683707,5089668,sra


In [66]:
response_handle = Entrez.epost(
    db = 'sra',
    id = ','.join(str(uid) for uid in srauid_map.uid)
)
web_env_info = Entrez.read(response_handle)

In [30]:
import re
import time
import urllib
from functools import partial


def extract_accessions(summary_xml, accession_summary, accession_matchers):
    for accession_type, accession_matcher in accession_matchers.items():
        match = accession_matcher.search(summary_xml['ExpXml'])
        accession = match.groups()[0] if match else None
        accession_summary[accession_type].append(accession)
    

def get_chunklimits(n_summaries, chunksize):
    retmin, retmax = 0, 0
    for i in range(0, n_summaries, chunksize):
        retmin = i 
        retmax = i + chunksize
        yield retmin, retmax

        
def retry(func, *args, n_retries = 5, sleep = 10, **kwargs):
    n_tries = 1
    success = False
    while not success:
        if n_tries == n_retries:
            raise RuntimeError('Exceeded retries!')
            
        try:
            response_handle = func(*args, **kwargs)
            success = True
            
        except urllib.error.HTTPError as e:
            print(e)
            print(f'current try: {n_tries}')
            n_tries += 1
        
    return response_handle
    
        
        
def parse_esummary_response(response_handle, accessions, accession_matchers):
    for summary in Entrez.read(response_handle):
        summary_accessions = extract_accessions(
            summary,
            accessions,
            accession_matchers
        )
        
    return accessions
        
    
def retrieve_summaries(web_env_info, n_summaries, accession_matchers, chunksize = 5000):
    get_sra_summaries = partial(
        Entrez.esummary,
        db = 'sra',
        WebEnv = web_env_info['WebEnv'],
        query_key = web_env_info['QueryKey']
    )
    accessions = {
        k: [] for k in accession_matchers.keys()
    }
    for retmin, retmax in get_chunklimits(n_summaries, chunksize):
        time.sleep(10) # avoid too many requests error
        response_handle = retry(
            get_sra_summaries,
            retstart = retmin,
            retmax = retmax
        )
        parse_esummary_response(
            response_handle,
            accessions,
            accession_matchers
        )
        
    return accessions

    
accession_matchers = {
    'experiment': re.compile('Experiment acc="([SRX0-9]+)'),
    'sample': re.compile('Sample acc="(SRS[0-9]+)'),
    'study': re.compile('Study acc="(SRP[0-9]+)'),
    'biosample': re.compile('Biosample>(SAMN[0-9]+)'),
    'bioproject': re.compile('Bioproject>(PRJNA[0-9]+)'),
    'geo': re.compile('name="(GSM[0-9]+)')
}

accessions = retrieve_summaries(
    web_env_info,
    len(srauid_map),
    accession_matchers,
    chunksize = 5000
)

accessions = pd.DataFrame().from_dict(accessions, orient = 'columns')
# somehow we get duplicates, suspect the post call
# because I did it twice so we might end up with 
# duplicated uids in the webenv but not sure
# anyway dropping duplicates here
accessions = accessions.drop_duplicates().reset_index(drop = True)

HTTP Error 429: Too Many Requests
current try: 1
HTTP Error 429: Too Many Requests
current try: 1
HTTP Error 502: Bad Gateway
current try: 1
HTTP Error 429: Too Many Requests
current try: 1
HTTP Error 429: Too Many Requests
current try: 1
HTTP Error 429: Too Many Requests
current try: 2
HTTP Error 429: Too Many Requests
current try: 1
HTTP Error 429: Too Many Requests
current try: 1


In [34]:
accessions = accessions.drop_duplicates().reset_index(drop = True)
accessions

,experiment,sample,study,biosample,bioproject,geo
0,SRX185895,SRS362050,SRP007525,SAMN01163911,PRJNA138597,GSM1000981
1,SRX185896,SRS362051,SRP007525,SAMN01163912,PRJNA138597,GSM1000982
2,SRX185897,SRS362052,SRP007525,SAMN01163913,PRJNA138597,GSM1000983
3,SRX185898,SRS362053,SRP007525,SAMN01163914,PRJNA138597,GSM1000984
4,SRX185899,SRS362054,SRP007525,SAMN01163915,PRJNA138597,GSM1000985
...,...,...,...,...,...,...
136338,SRX3155087,SRS2486922,SRP116913,SAMN07596953,PRJNA401870,GSM2770367
136339,SRX3155088,SRS2486921,SRP116913,SAMN07596952,PRJNA401870,GSM2770368
136340,SRX3155089,SRS2486924,SRP116913,SAMN07596951,PRJNA401870,GSM2770369
136341,SRX3155090,SRS2486923,SRP116913,SAMN07596950,PRJNA401870,GSM2770370


In [37]:
def determine_accession_type(accession):
    types = {
        'experiment': 'SRX',
        'biosample': 'SAMN',
        'geo': 'GSM'
    }
    
    for accession_type, accession_prefix in types.items():
        if accession.startswith(accession_prefix):
            return accession_type
    

srauid_map['accession_type'] = srauid_map.accession.apply(
    lambda x: determine_accession_type(x)
)
srauid_map

,accession,uid,database,accession_type
0,SRX185895,243394,sra,experiment
1,SRX185896,243395,sra,experiment
2,SRX185897,243396,sra,experiment
3,SRX185898,243397,sra,experiment
4,SRX185899,243398,sra,experiment
...,...,...,...,...
136578,SRX3155087,4452724,sra,experiment
136579,SRX3155088,4452725,sra,experiment
136580,SRX3155089,4452726,sra,experiment
136581,SRX3155090,4452727,sra,experiment


In [39]:
merged_groups = []
for accession_type, group in srauid_map.groupby('accession_type'):
    group = group.rename(
        columns = {'accession': accession_type}
    )
    
    merged_group = group.merge(
        accessions,
        on = accession_type,
        how = 'inner'
    )
    merged_groups.append(merged_group)
    
merged = pd.concat(merged_groups)
merged

,biosample,uid,database,accession_type,experiment,sample,study,bioproject,geo
0,SAMN02383463,2398516,sra,biosample,SRX1671895,SRS1369054,SRP031885,PRJNA224140,GSM1249874
1,SAMN02383461,2398517,sra,biosample,SRX1671896,SRS1369053,SRP031885,PRJNA224140,GSM1249875
2,SAMN02383456,2398518,sra,biosample,SRX1671897,SRS1369052,SRP031885,PRJNA224140,GSM1249876
3,SAMN02383457,2398519,sra,biosample,SRX1671898,SRS1369051,SRP031885,PRJNA224140,GSM1249877
4,SAMN02383460,2398520,sra,biosample,SRX1671899,SRS1369050,SRP031885,PRJNA224140,GSM1249878
...,...,...,...,...,...,...,...,...,...
133003,SAMN07596953,4452724,sra,experiment,SRX3155087,SRS2486922,SRP116913,PRJNA401870,GSM2770367
133004,SAMN07596952,4452725,sra,experiment,SRX3155088,SRS2486921,SRP116913,PRJNA401870,GSM2770368
133005,SAMN07596951,4452726,sra,experiment,SRX3155089,SRS2486924,SRP116913,PRJNA401870,GSM2770369
133006,SAMN07596950,4452727,sra,experiment,SRX3155090,SRS2486923,SRP116913,PRJNA401870,GSM2770370


In [43]:
merged.to_csv(
    'sample_accessions.tsv',
    sep = '\t',
    index = False
)

In [78]:
list(it.product(list('A'), [{'id': 1}, {'id': 2}]))

[('A', {'id': 1}), ('A', {'id': 2})]

In [80]:
# eLink does not work for id lists larger than 1000
# see this issue https://github.com/biopython/biopython/issues/1559
# need to batch them locally
import itertools as it

def get_links(link_item):
    dbto_ids = link_item['LinkSetDb'][0]['Link']
    dbfrom_ids = link_item['IdList']
    
    links = [
        [dbfrom_id, dbto_id['Id']] for dbfrom_id, dbto_id in it.product(dbfrom_ids, dbto_ids)
    ]
    
    return links
        

def parse_elink_response(elink_response_handle):
    elink_response = Entrez.read(elink_response_handle)
    linked_ids = []
    for link_item in elink_response:
        links = get_links(
            link_item
        )
        linked_ids.extend(links)
    
    return linked_ids
    

def link_uids(uid_list, dbfrom, dbto):
    linked_ids = []
    for id_chunk in it.batched(uid_list, n = 1000):
        response_handle = Entrez.elink(
            dbfrom = dbfrom,
            db = dbto,
            id = id_chunk
        )
        
        links = parse_elink_response(
            response_handle
        )
    
        linked_ids.extend(
            links
        )
    
    linked_ids = pd.DataFrame(
        linked_ids,
        columns = [dbfrom, dbto]
    )
    
    return linked_ids


link_uids(
    srauid_map.uid, 
    dbfrom = 'sra',
    dbto = 'biosample'
)

,sra,biosample
0,243394,1163911
1,243395,1163912
2,243396,1163913
3,243397,1163914
4,243398,1163915
...,...,...
995,424434,2199431
996,424435,2199428
997,424436,2199430
998,424437,2199434
